<a href="https://colab.research.google.com/github/florence-aikins/car-price-predictions/blob/main/car_price_predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing libraries
%matplotlib Inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set(
    { "figure.figsize": (6, 4) },
    style='ticks',
    color_codes=True,
    font_scale=0.8
)
%config InlineBackend.figure_formats = set(('retina', 'svg'))

import sklearn
sklearn.set_config(transform_output="pandas")

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.tree import export_text
from sklearn.metrics import PredictionErrorDisplay
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance

from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.base import clone
from sklearn.model_selection import cross_validate

from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay
from sklearn.inspection import partial_dependence
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.feature_selection import RFECV

from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from ipywidgets import interact
from sklearn.manifold import Isomap

from sklearn.cluster import KMeans
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install shap --upgrade -q
import shap
shap.initjs()

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/car_registration/car_registration.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.groupby("price").size()

In [ ]:
df = df.drop('public_reference', axis=1)
df = df.drop('reg_code', axis=1)
df = df.drop('crossover_car_and_van', axis=1)
df = df.drop('standard_colour', axis=1)
df = df.drop('standard_model', axis=1)
df.head()

In [ ]:
sns.histplot(x=df["price"], bins=50)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sns.boxplot(x=df["mileage"])
plt.title("Mileage Distribution")
plt.xlabel("Mileage")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sns.histplot(x=df["year_of_registration"], bins=50)
plt.title("Year of Registration Distribution")
plt.xlabel("Year of Registration")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df['fuel_type'].value_counts().plot(kind='bar')
plt.title('Fuel Type Distribution')
plt.xlabel('Fuel Type')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.countplot(x='vehicle_condition', data=df)
plt.title('vehicle condition Distribution')
plt.xlabel('vehicle condition')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['standard_make'].value_counts().head(20).plot(kind='bar')
plt.title('Standard Make Distribution')
plt.xlabel('Standard Make')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['body_type'].value_counts().plot(kind='bar')
plt.title('Body Type Distribution')
plt.xlabel('Body Type')
plt.ylabel('Frequency')
plt.show()

In [ ]:
corr = df[['price', 'mileage', 'year_of_registration']].corr()
sns.heatmap(corr, annot=True, cmap = 'coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
sns.boxplot(y='fuel_type', x='price', data=df)
plt.title('Fuel Type vs Price')
plt.show()

In [ ]:
sns.barplot(y='fuel_type', x='price', data=df)
plt.title('Fuel Type vs Price')
plt.show()

In [ ]:
sns.barplot(y='body_type', x='price', data=df)
plt.title('Body Type vs Price')
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
df['mileage'] = df['mileage'].fillna(df['mileage'].median())
df['year_of_registration'] = df['year_of_registration'].fillna(df['year_of_registration'].median())
df = df[df['year_of_registration']>= 1903]
df['fuel_type'] = df['fuel_type'].fillna(df['fuel_type'].mode())
df['body_type'] = df['body_type'].fillna(df['body_type'].mode())

In [ ]:
df['price'] = df['price'].astype(int)
df['mileage'] = df['mileage'].astype(int)
df['year_of_registration'] = df['year_of_registration'].astype(int)

In [ ]:
df = df[df['price']>500]
df = df[df['mileage']>0]

In [ ]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [ ]:
df['price'] = df['price'].clip(lower,upper)

In [ ]:
Q1 = df['mileage'].quantile(0.25)
Q3 = df['mileage'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [ ]:
df['mileage'] = df['mileage'].clip(lower,upper)

In [ ]:
df.head()

In [ ]:
top_makes = df['standard_make'].value_counts().nlargest(20).index
df['standard_make'] = df['standard_make'].where(df['standard_make'].isin(top_makes), 'Other')

In [ ]:
df['vehicle_condition'] = df['vehicle_condition'].astype(str).str.strip().str.upper()
df['vehicle_condition'] = df['vehicle_condition'].map({'USED': 0, 'NEW': 1})
df['vehicle_condition'] = df['vehicle_condition'].astype(int)

In [ ]:
df.head()

In [ ]:
target = 'price'
numerical_cols = ['mileage', 'year_of_registration']
categorical_cols = ['fuel_type', 'body_type', 'standard_make']
binary_cols = ['vehicle_condition']

In [ ]:
ohe = OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore')

In [ ]:
X = df.drop(target, axis=1)
y = df[target]

**PIPELINE**

In [ ]:
def create_pp_ppln(X, linear_model=False):

  numerical_features = X.select_dtypes(exclude='object').columns.tolist()
  numerical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="mean"))
        ])

  if linear_model:
    numerical_transformer.steps.extend([
            ("scaler", StandardScaler()),
        ])
  categorical_features = X.select_dtypes(include='object').columns.tolist()
  categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encode", ohe),
        ]
    )

  preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )

  return preprocessor

In [ ]:
def create_regr_ppln(est, X, linear_model=False):
    """ """
    regr_pipe = Pipeline(
        steps=[
            ("pp", create_pp_ppln(X, linear_model)),
            ("regr", est)
        ]
    )

    return regr_pipe

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
   X, y, test_size=0.3,
    random_state = 42
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    random_state = 42
)

In [ ]:
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
X_sample = X_train.sample(n=50000, random_state=42)
y_sample = y_train[X_sample.index]

In [ ]:
Xt_sample = X_test.sample(n=50000, random_state=42)
yt_sample = y_test[Xt_sample.index]

In [ ]:
Xv_sample = X_val.sample(n=50000, random_state=42)
yv_sample = y_val[Xv_sample.index]

**LINEAR** **REGRESSION**

In [ ]:
lr   = create_regr_ppln(LinearRegression(), X_sample, linear_model=True)

In [ ]:
lr.fit(X_sample, y_sample)

In [ ]:
lr.score(X_sample, y_sample)

In [ ]:
lr.score(Xt_sample, yt_sample)

In [ ]:
lr.score(Xv_sample, yv_sample)

In [ ]:
cross_val_score(lr, X_sample, y_sample, cv=5)

In [ ]:
scores = cross_val_score(lr, X_sample, y_sample, cv=5)
scores.mean(), scores.std()

**AUTOMATED** **FEATURE** **SELECTION**

In [ ]:
# Create the preprocessor using the function and fit it to the full X data
preprocessor_for_rfe = create_pp_ppln(X, linear_model=True)
X_processed_for_rfe = preprocessor_for_rfe.fit_transform(X)

rfe_selector = RFECV(estimator=LinearRegression(), step=1, cv=5)

In [ ]:
# Fit the RFE selector on the preprocessed numerical data
rfe_selector.fit(X_processed_for_rfe, y)

In [ ]:
X_sel = rfe_selector.transform(X_processed_for_rfe)
X_sel.head()

In [ ]:
rfe_selector.get_feature_names_out()

In [ ]:
n_scores = len(rfe_selector.cv_results_["mean_test_score"])
n_scores

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.errorbar(
    range(1, n_scores+1),
    rfe_selector.cv_results_["mean_test_score"],
    yerr=rfe_selector.cv_results_["std_test_score"],
)
ax.set_xlabel("Number of features selected")
ax.set_ylabel("Mean test score");

In [ ]:
rfe_selector.ranking_

In [ ]:
np.argsort(rfe_selector.ranking_)

In [ ]:
np.flip(np.argsort(rfe_selector.ranking_))

In [ ]:
rfe_selector.feature_names_in_[ np.flip(np.argsort(rfe_selector.ranking_)) ]

In [ ]:
rfe_selector.support_

**RANDOM** **FOREST**

In [ ]:
rfr  = create_regr_ppln(RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, min_samples_split=5, min_samples_leaf = 2, random_state=42), X_sample)

In [ ]:
rfr.fit(X_sample, y_sample)

In [ ]:
rfr.score(Xt_sample, yt_sample)

In [ ]:
rfr.score(X_sample, y_sample)

In [ ]:
scores = cross_val_score(rfr, X_sample, y_sample, cv=5)
scores.mean(), scores.std()

**GRADIENT** **BOOSTING**

In [ ]:
gbr = create_regr_ppln(GradientBoostingRegressor(random_state = 42), X_sample)

In [ ]:
gbr.fit(X_sample, y_sample)

In [ ]:
gbr.score(Xt_sample, yt_sample)

In [ ]:
gbr.score(X_sample, y_sample)

In [ ]:
scores = cross_val_score(gbr, X_sample, y_sample, cv=5)
scores.mean(), scores.std()

In [ ]:
my_models = [ gbr, rfr, lr ]

In [ ]:
X_data_sample = X.sample(n=50000, random_state=42)
y_data_sample = y[X_sample.index]
model_results_list = []
for my_model in my_models:
    eval_results = cross_validate(
        my_model, X_data_sample, y_data_sample, cv=5,
        scoring='neg_mean_absolute_error',
        return_train_score=True
    )
    model_results_list.append(
        (-eval_results['test_score'].mean(), eval_results['test_score'].std(),
         -eval_results['train_score'].mean(), eval_results['train_score'].std())
    )

In [ ]:
model_results = pd.DataFrame(
    model_results_list,
    columns=['test_mae_mean', 'test_mae_std', 'train_mae_mean', 'train_mae_std'],
    index=['gbr', 'rfr', 'lr']
)

In [ ]:
model_results

**ENSEMBLING**

In [ ]:
for est in my_models:
    est.fit(X_sample, y_sample)

In [ ]:
ensemble = VotingRegressor(
   estimators = [
        ("gb", gbr),
        ("rf", rfr),
        ('lr', lr)
    ]
)
ensemble.fit(X_sample, y_sample)

In [ ]:
eval_results = cross_validate(
    ensemble, X_data_sample, y_data_sample, cv=5,
    scoring='neg_mean_absolute_error',
    return_train_score=True
)

In [ ]:
ensemble_result = (
    -eval_results['test_score'].mean(), eval_results['test_score'].std(),
    -eval_results['train_score'].mean(), eval_results['train_score'].std()
)

In [ ]:
model_results.loc['ensemble'] = ensemble_result

In [ ]:
model_results

**PREDICTION**

In [ ]:
Xt = X.head(50000)
yt = y.head(50000)
rfr_pred = rfr.predict(Xt)
gbr_pred = gbr.predict(Xt)
lr_pred = lr.predict(Xt)
ens_pred = ensemble.predict(Xt)
print('rf_pred is: ', rfr_pred)
print('gb_pred is: ', gbr_pred)
print('lr_pred is: ', lr_pred)
print('ens_pred is: ', ens_pred)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

idx = np.random.choice(len(yt), 2000, replace=False)

plt.scatter(yt.iloc[idx], rfr_pred[idx], label='Random Forest')
plt.scatter(yt.iloc[idx], gbr_pred[idx], label='Gradient Boosting')
plt.scatter(yt.iloc[idx], lr_pred[idx], label='Linear Regression')
plt.scatter(yt.iloc[idx], ens_pred[idx], label='Ensemble')

plt.plot([yt.iloc[idx].min(), yt.iloc[idx].max()],
         [yt.iloc[idx].min(), yt.iloc[idx].max()],
         'k--', lw=2)

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Prices')
plt.legend()
plt.show()

In [ ]:
plt.scatter(yt.iloc[idx], ens_pred[idx], label='Ensemble')

plt.plot([yt.iloc[idx].min(), yt.iloc[idx].max()],
         [yt.iloc[idx].min(), yt.iloc[idx].max()],
         'k--', lw=2)

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Prices')
plt.legend()
plt.show()

**BEST** **MODEL**

In [ ]:
#best model is random forest
rfr  = create_regr_ppln(RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, min_samples_split=5, min_samples_leaf = 2, random_state=42), X_train)

In [ ]:
rfr.fit(X_train, y_train)

In [ ]:
def plot_est_feat_imp_barh(est, feat_names, ax=None, top_feat_k=10, style_kws={}):
    """ """
    if ax is None:
        fig, ax = plt.subplots()

    return pd.Series(
        est.feature_importances_,
        index=feat_names
    ).sort_values().tail(top_feat_k).plot.barh(**style_kws)

In [ ]:
plot_est_feat_imp_barh(rfr['regr'], rfr['pp'].get_feature_names_out());

**PERMUTATION** **IMPORTANCE**

In [ ]:
perm_importance = permutation_importance(
    rfr, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

In [ ]:
perm_importance_df = pd.DataFrame(
    dict(
        feature=X_test.columns,
        pi_mean=perm_importance['importances_mean'],
        pi_std=perm_importance['importances_std']
    )
)

In [ ]:
perm_importance_df

In [ ]:
perm_importance_df_sorted = perm_importance_df.sort_values(by='pi_mean', ascending=True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(
    perm_importance_df_sorted['pi_mean'],
    perm_importance_df_sorted['feature'],
    xerr=perm_importance_df_sorted['pi_std'],
    fmt='o', color='black',
    ecolor='lightgray', elinewidth=3, capsize=0
)

ax.set_ylabel('Feature')
ax.set_xlabel('Permutation Importance Mean');

**SHAP**

In [ ]:
X_shap_sample = X_train.sample(n=1000, random_state=42)
Xt_shap_sample = X_test.sample(n=1000, random_state=42)
y_shap_sample = y_train[X_shap_sample.index]
yt_shap_sample = y_test[Xt_shap_sample.index]

In [ ]:
X_train_pp = rfr['pp'].transform(X_shap_sample)
column_names = X_train_pp.columns
X_test_pp  = rfr['pp'].transform(Xt_shap_sample)

In [ ]:
X_processed = rfr.named_steps['pp'].transform(X_shap_sample)

In [ ]:
explainer = shap.Explainer(rfr.named_steps['regr'], X_processed)
explainer

In [ ]:
X_train_pp.head(1)

In [ ]:
X_test_pp.head(1)

In [ ]:
explanations = explainer(X_test_pp, check_additivity=False)
explanations

In [ ]:
type(explanations)

In [ ]:
explanations.shape

In [ ]:
cls_idx = 1
row_idx = 0
cls_idx, row_idx

In [ ]:
shap.plots.waterfall(explanations[row_idx])

In [ ]:
X_test_pp.iloc[[row_idx]]

In [ ]:
rfr.predict(Xt_shap_sample.iloc[[row_idx]])

In [ ]:
shap_values = explainer(X_processed[:300])

In [ ]:
shap_values.shape

In [ ]:
y_shap_sample.unique()

In [ ]:
shap_values_df = pd.DataFrame(
    shap_values.values, columns=column_names, index=Xt_shap_sample.index[:300]
)

In [ ]:
shap_values_df.head()

In [ ]:
X_test_pp.head()

In [ ]:
shap.plots.beeswarm(explanations)

In [ ]:
shap.summary_plot(explanations)

In [ ]:
shap_values_df.query("year_of_registration<-0.2")

In [ ]:
Xt_shap_sample.loc[shap_values_df.query("year_of_registration<-0.2").index]

In [ ]:
Xt_shap_sample['year_of_registration'].describe()

In [ ]:
shap.summary_plot(explanations, plot_type="bar")

**PARTIAL** **DEPENDENCY**

In [ ]:
plot_est_feat_imp_barh(rfr['regr'], rfr['pp'].get_feature_names_out());

In [ ]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)
PartialDependenceDisplay.from_estimator(
    rfr, X_test, features=['year_of_registration', 'mileage'],
    kind='average',
    subsample=100, grid_resolution=30, n_jobs=-1, random_state=42,
    ax=ax, n_cols=2
);

In [ ]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)
PartialDependenceDisplay.from_estimator(
    rfr, X_test, features=['year_of_registration', 'mileage'],
    kind='both', centered=True,
    subsample=100, grid_resolution=30, n_jobs=-1, random_state=42,
    ax=ax, n_cols=2, pd_line_kw={'color': 'red'}
);

**PCA**

In [ ]:
pca_pipeline = Pipeline([
    ('pp', create_pp_ppln(X, linear_model=True)),
    ('pca', PCA())
])

In [ ]:
X_pca = pca_pipeline.fit_transform(X)
print(X_pca.shape)

In [ ]:
pca = pca_pipeline.named_steps['pca']
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

In [ ]:
print("Explained variance for 13 components:")
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var), 1):
    print(f"PC{i}: {ev:.3f} (Cumulative: {cv:.3f})")

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(range(1, 14), explained_var[:13], alpha=0.7, label='Explained variance')
plt.plot(range(1, 14), cumulative_var[:13], 'ro-', label='Cumulative variance')
plt.xlabel('Principal component')
plt.ylabel('Variance explained')
plt.xticks(range(1, 14))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pca_pipeline = Pipeline([
    ('pp', create_pp_ppln(X, linear_model=True)),
    ('pca', PCA(n_components=0.95))
])
X_pca = pca_pipeline.fit_transform(X)
print(X_pca.shape)

In [ ]:
lr_pca = LinearRegression()

In [ ]:
scores_pca = cross_val_score(lr_pca, X_pca, y, cv=5)
scores_pca.mean(), scores_pca.std()

**ISOMAP**

In [ ]:
X_sub = X.sample(n=5000, random_state=42)
y_sub = y.loc[X_sub.index]
iso_pipeline = Pipeline([
    ('pp', create_pp_ppln(X, linear_model=True)),
    ('iso', Isomap(n_neighbors=10, n_components=10))
])

In [ ]:
X_iso = iso_pipeline.fit_transform(X_sub)
print(X_iso.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_iso.iloc[:, 0], X_iso.iloc[:, 1], c=y_sub, cmap='viridis', s=20)
plt.colorbar(label='Price')
plt.show()

In [ ]:
lr_iso = LinearRegression()
scores_iso = cross_val_score(lr_iso, X_iso, y_sub, cv=5)
print(scores_iso.mean(), scores_iso.std())

**POLYNOMIAL** **REGRESSION**

In [ ]:
poly_model=Pipeline([
    ('pp', create_pp_ppln(X, linear_model=True)),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('regr', LinearRegression())
])

In [ ]:
scorres_poly = cross_val_score(poly_model, X[:5000], y[:5000], cv=3)
print(scorres_poly.mean(), scorres_poly.std())

In [ ]:
X_sample_single_feature = X['mileage'].sample(n=1000, random_state=42)
y_sample_single_feature = y.loc[X_sample_single_feature.index]

X_poly = X_sample_single_feature.values.reshape(-1, 1)
y_poly = y_sample_single_feature.values

models_poly = {}
for degree in [1, 2, 3, 5]:
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_transformed = poly_features.fit_transform(X_poly)
    model = LinearRegression()
    X_poly_transformed_sm = sm.add_constant(X_poly_transformed)
    model.fit(X_poly_transformed_sm, y_poly)
    models_poly[degree] = model

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

# Plot 1: Linear fit (degree 1)
ax = axes[0, 0]
# Create an X range for plotting the curve
X_test_range = np.linspace(X_poly.min(), X_poly.max(), 200).reshape(-1, 1)
poly_features_1 = PolynomialFeatures(degree=1, include_bias=False)
X_test_poly_transformed_1 = poly_features_1.fit_transform(X_test_range)
X_test_sm_1 = sm.add_constant(X_test_poly_transformed_1)
y_pred_1 = models_poly[1].predict(X_test_sm_1)

ax.scatter(X_poly, y_poly, alpha=0.5, s=30, color='gray', label="Data points")
ax.plot(X_test_range, y_pred_1, color='red', linewidth=2.5, label="Fit (degree 1)")
ax.set_title("Linear Model: Underfitting", fontweight='bold', fontsize=11)
ax.set_xlabel("Mileage")
ax.set_ylabel("Price")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 2: Quadratic fit (degree 2)
ax = axes[0, 1]
poly_features_2 = PolynomialFeatures(degree=2, include_bias=False)
X_test_poly_transformed_2 = poly_features_2.fit_transform(X_test_range)
X_test_sm_2 = sm.add_constant(X_test_poly_transformed_2)
y_pred_2 = models_poly[2].predict(X_test_sm_2)

ax.scatter(X_poly, y_poly, alpha=0.5, s=30, color='gray', label="Data points")
ax.plot(X_test_range, y_pred_2, color='green', linewidth=2.5, label="Fit (degree 2)")
ax.set_title("Quadratic Model: Better fit", fontweight='bold', fontsize=11)
ax.set_xlabel("Mileage")
ax.set_ylabel("Price")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 3: Cubic fit (degree 3)
ax = axes[1, 0]
poly_features_3 = PolynomialFeatures(degree=3, include_bias=False)
X_test_poly_transformed_3 = poly_features_3.fit_transform(X_test_range)
X_test_sm_3 = sm.add_constant(X_test_poly_transformed_3)
y_pred_3 = models_poly[3].predict(X_test_sm_3)

ax.scatter(X_poly, y_poly, alpha=0.5, s=30, color='gray', label="Data points")
ax.plot(X_test_range, y_pred_3, color='purple', linewidth=2.5, label="Fit (degree 3)")
ax.set_title("Cubic Model: Best Fit", fontweight='bold', fontsize=11)
ax.set_xlabel("Mileage")
ax.set_ylabel("Price")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 4: Comparison of model curves
ax = axes[1, 1]
poly_features_5 = PolynomialFeatures(degree=5, include_bias=False)
X_test_poly_transformed_5 = poly_features_5.fit_transform(X_test_range)
X_test_sm_5 = sm.add_constant(X_test_poly_transformed_5)
y_pred_5 = models_poly[5].predict(X_test_sm_5)

ax.scatter(X_poly, y_poly, alpha=0.25, s=20, color='gray', label='Data points')
ax.plot(X_test_range, y_pred_1, linewidth=2, color='red', label='Fit (degree 1)')
ax.plot(X_test_range, y_pred_2, linewidth=2, color='green', label='Fit (degree 2)')
ax.plot(X_test_range, y_pred_3, linewidth=2, color='purple', label='Fit (degree 3)')
ax.plot(X_test_range, y_pred_5, linewidth=2, color='orange', label='Fit (degree 5)')
ax.set_title("Model Curve Comparison", fontweight='bold', fontsize=11)
ax.set_xlabel("Mileage")
ax.set_ylabel("Price")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.xticks(rotation=45)
plt.show()

**CLUSTERING** **FOR** **FEATURE** **ENGINEERING**

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)

In [ ]:
preprocessor = create_pp_ppln(X, linear_model=True)
X_scaled = preprocessor.fit_transform(X)
kmeans.fit(X_scaled)

In [ ]:
clusters = kmeans.fit_predict(X_scaled)
print(clusters)

In [ ]:
X_with_cluster = X.copy()
X_with_cluster['cluster'] = clusters

In [ ]:
rfr_clusters = create_regr_ppln(RandomForestRegressor(), X_with_cluster)

In [ ]:
scores_cluster = cross_val_score(rfr_clusters[:5000], X_with_cluster[:5000], y[:5000], cv=3, n_jobs=-1)
print(scores_cluster.mean(), scores_cluster.std())

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_scaled.iloc[:,0], y=X_scaled.iloc[:,1], c=clusters)
plt.title("K-Means Clustering")
plt.show()